# Week 4 -- Function 2: Neural Network Surrogate + Bayesian Optimisation (2D)

In [ ]:
# -- WEEKLY OBSERVATIONS LOG --------------------------------------------------------------------------------
import numpy as np

INITIAL_SIZE = 10
DATA_PATH_X  = '../data/function_2/initial_inputs.npy'
DATA_PATH_Y  = '../data/function_2/initial_outputs.npy'

weekly_log = [
    ([0.000000, 1.000000], 0.02486946982716038),          # Week 1: UCB beta=2.5 -- received 0.024869
    ([1.000000, 0.326531], 0.14927005994804756),           # Week 2: UCB beta=3.5 -- received 0.149270 (IMPROVED)
    ([1.000000, 1.000000], 0.03643861312405127),           # Week 3: UCB beta=3.5 -- received 0.036439 (corner dead)
    # Week 4: add next week's result here as ([x1, x2], y_value)
]

X_disk = np.load(DATA_PATH_X)
Y_disk = np.load(DATA_PATH_Y)

n_missing = (INITIAL_SIZE + len(weekly_log)) - len(Y_disk)
if n_missing > 0:
    new_entries = weekly_log[len(weekly_log) - n_missing:]
    for x_vals, y_val in new_entries:
        X_disk = np.vstack([X_disk, np.array([x_vals])])
        Y_disk = np.append(Y_disk, y_val)
    np.save(DATA_PATH_X, X_disk)
    np.save(DATA_PATH_Y, Y_disk)
    print(f'Synced {n_missing} new observation(s).')
else:
    print('Already up-to-date.')

print(f'X: {X_disk.shape} | Y: {Y_disk.shape}')
current_week = len(Y_disk) - INITIAL_SIZE + 1
print(f'Week {current_week} | {len(Y_disk)} total observations ({INITIAL_SIZE} initial + {len(Y_disk)-INITIAL_SIZE} added)')

In [ ]:
# Cell 3: Load data and inspect
# Function 2: Noisy ML log-likelihood, 2D input, Maximise

X = np.load('../data/function_2/initial_inputs.npy')
Y = np.load('../data/function_2/initial_outputs.npy')

print(f'Input  shape : {X.shape}   (n_samples x n_dimensions)')
print(f'Output shape : {Y.shape}  (n_samples,)')
print()

pairs = sorted(zip(Y.tolist(), X.tolist()), reverse=True)
Y_sorted = [p[0] for p in pairs]
X_sorted = [p[1] for p in pairs]

print('=' * 72)
print('  All observations (sorted descending by Y)')
print('=' * 72)
for i, (y_val, x_val) in enumerate(zip(Y_sorted, X_sorted)):
    marker = '  <-- best' if i == 0 else ''
    print(f'  [{i+1:2d}]  X = [{x_val[0]:.6f}, {x_val[1]:.6f}]   Y = {y_val:+.6f}{marker}')
print('=' * 72)

best_Y = Y_sorted[0]
best_X = np.array(X_sorted[0])
n_dims = X.shape[1]
print(f'\n  Best Y*  = {best_Y:.6f}')
print(f'  Best X*  = [{best_X[0]:.8f}, {best_X[1]:.8f}]')

In [ ]:
# Cell 4: Fit GP with log-transformed Y (kept for comparison and UCB fallback)
import warnings
warnings.filterwarnings('ignore')
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

Y_fit = np.log(np.abs(Y) + 1e-300)

kernel = RBF(length_scale=0.1, length_scale_bounds='fixed')
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10)
gp.fit(X, Y_fit)

print('=' * 62)
print('  GP Fitting Results')
print('=' * 62)
print(f'  Kernel                   : {gp.kernel_}')
print(f'  Log-marginal-likelihood  : {gp.log_marginal_likelihood_value_:.4f}')
pred_mean, pred_std = gp.predict(best_X.reshape(1, -1), return_std=True)
actual_log = np.log(np.abs(best_Y) + 1e-300)
print(f'  Sanity check at best X*  : GP mean={pred_mean[0]:.4f}  actual={actual_log:.4f}  std={pred_std[0]:.6f}')
print('=' * 62)

In [ ]:
# Cell 5: Neural Network Surrogate Model (TensorFlow/Keras)
# Following course Assignment 15.1 (Sequential API) + Assignment 15.2 (GradientTape training).

import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

tf.random.set_seed(42)

Y_log    = np.log(np.abs(Y) + 1e-300)
y_scaler = StandardScaler()
Y_scaled = y_scaler.fit_transform(Y_log.reshape(-1, 1)).ravel().astype(np.float32)

X_t = tf.constant(X.astype(np.float32))
Y_t = tf.constant(Y_scaled)

# Build surrogate using Sequential API (Assignment 15.1 style)
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu',
                          kernel_regularizer=tf.keras.regularizers.l2(1e-3),
                          input_shape=(n_dims,)),
    tf.keras.layers.Dropout(0.1),
    tf.keras.layers.Dense(64, activation='relu',
                          kernel_regularizer=tf.keras.regularizers.l2(1e-3)),
    tf.keras.layers.Dropout(0.1),
    tf.keras.layers.Dense(1)
], name='surrogate_nn')

# Loss function and optimizer (Assignment 15.2 style)
loss_fn   = tf.keras.losses.MeanSquaredError()
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)

# train_step using GradientTape (Assignment 15.2 style)
@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        y_pred = tf.squeeze(model(x, training=True))
        loss   = loss_fn(y, y_pred) + tf.add_n(model.losses)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return loss

# Training loop (Assignment 15.2 style)
for epoch in range(1500):
    loss = train_step(X_t, Y_t)
    if epoch % 300 == 0:
        print(f'  Epoch {epoch:4d}  MSE (scaled) = {loss.numpy():.4f}')

print(f'\n  Final training MSE (scaled): {loss.numpy():.4f}')

nn_pred_best = float(tf.squeeze(model(tf.expand_dims(
    tf.constant(best_X.astype(np.float32)), 0), training=False)))
print(f'  NN prediction at best X*: {nn_pred_best:.4f} (scaled)')

In [ ]:
# Cell 6: Gradient-Guided Acquisition with Trust-Region (TensorFlow)
# Assignment 15.2 style: tf.Variable + tf.GradientTape for gradient ascent on input.
# F2 best is [1.0, 0.326531] -- gradient should refine x2.

import numpy as np

trust_radius = 0.20   # F2: exploit high-x1 strip; ±0.20 allows x2 refinement

# --- Strategy A: NN gradient ascent (Assignment 15.2 style) ---
lo = np.clip(best_X - trust_radius, 0.0, 1.0).astype(np.float32)
hi = np.clip(best_X + trust_radius, 0.0, 1.0).astype(np.float32)

x_opt     = tf.Variable(best_X.copy().astype(np.float32))
opt_inner = tf.keras.optimizers.Adam(learning_rate=5e-3)

for step in range(800):
    with tf.GradientTape() as tape:
        score      = tf.squeeze(model(tf.expand_dims(x_opt, 0), training=False))
        loss_inner = -score
    gradients = tape.gradient(loss_inner, [x_opt])
    opt_inner.apply_gradients(zip(gradients, [x_opt]))
    x_opt.assign(tf.clip_by_value(x_opt, lo, hi))

grad_candidate = x_opt.numpy().copy()

# --- Strategy B: GP UCB ---
beta = 2.5   # moderate exploitation for F2's known good region
np.random.seed(42)
X_grid = np.random.uniform(0, 1, size=(30000, n_dims))
post_mean, post_std = gp.predict(X_grid, return_std=True)
ucb_scores = post_mean + beta * post_std
gp_candidate = X_grid[np.argmax(ucb_scores)]

# --- Pick winner ---
nn_score_grad = float(tf.squeeze(model(tf.expand_dims(
    tf.constant(grad_candidate.astype(np.float32)), 0), training=False)))
nn_score_gp   = float(tf.squeeze(model(tf.expand_dims(
    tf.constant(gp_candidate.astype(np.float32)), 0), training=False)))

selected = 'NN gradient ascent' if nn_score_grad >= nn_score_gp else 'GP UCB'
next_x   = grad_candidate if nn_score_grad >= nn_score_gp else gp_candidate

portal_string = '-'.join([f'{v:.6f}' for v in next_x])

print('=' * 72)
print('  Acquisition Results')
print('=' * 72)
print(f'  Trust-region radius     : ±{trust_radius} per dimension')
print(f'  NN gradient candidate : [{grad_candidate[0]:.6f}, {grad_candidate[1]:.6f}]  (NN score: {nn_score_grad:.4f})')
print(f'  GP UCB  candidate     : [{gp_candidate[0]:.6f}, {gp_candidate[1]:.6f}]  (NN score: {nn_score_gp:.4f})')
print(f'  Selected method       : {selected}')
print()
print(f'  Next query point      : [{next_x[0]:.6f}, {next_x[1]:.6f}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 72)

In [ ]:
# Cell 7: Input Gradient Sensitivity Analysis (TensorFlow)
# Assignment 15.2 style: tf.Variable watched by tf.GradientTape to compute ∂output/∂x_i.

x_sens = tf.Variable(best_X.copy().astype(np.float32))

with tf.GradientTape() as tape:
    out = tf.squeeze(model(tf.expand_dims(x_sens, 0), training=False))

grads = np.abs(tape.gradient(out, x_sens).numpy())

print('Input gradient magnitudes at best X*:')
print(f'  (best X* = [{best_X[0]:.6f}, {best_X[1]:.6f}],  Y* = {best_Y:.6f})')
print()
max_g = grads.max() if grads.max() > 0 else 1.0
for i, g in enumerate(grads):
    bar = '#' * max(1, int(g * 40 / max_g))
    print(f'  x{i+1}: |∂f/∂x{i+1}| = {g:.4f}  {bar}')

most_sensitive = int(np.argmax(grads)) + 1
print(f'\n  Most sensitive dimension: x{most_sensitive}')
print(f'  Interpretation: the NN surrogate predicts the output is most responsive to x{most_sensitive} near the current best.')

In [ ]:
# Cell 7b: SVM Classification Check (secondary validation)
# Label top-30% as 'good', train RBF-SVM, check proposed query classification.

from sklearn.svm import SVC

threshold = np.percentile(Y, 70)   # top 30% = 'good'
labels = (Y >= threshold).astype(int)

print(f'  Classification threshold (70th percentile of Y): {threshold:.6f}')
print(f'  Good samples: {labels.sum()} / {len(labels)}')
print()

if labels.sum() >= 2 and (labels == 0).sum() >= 2:
    svm_clf = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
    svm_clf.fit(X, labels)

    prob_best  = svm_clf.predict_proba(best_X.reshape(1, -1))[0, 1]
    prob_query = svm_clf.predict_proba(next_x.reshape(1, -1))[0, 1]

    print(f'  SVM support vectors    : {svm_clf.n_support_} (per class)')
    print(f'  P(good) at best X*     : {prob_best:.3f}  (reference)')
    print(f'  P(good) at next query  : {prob_query:.3f}')
    print()
    if prob_query >= 0.5:
        print('  ✓ SVM endorses next query as likely good')
    elif prob_query >= 0.3:
        print('  ~ SVM uncertain -- query is near the decision boundary (informative region)')
    else:
        print('  ✗ SVM flags query as likely poor -- consider revising or accepting exploration risk')
else:
    print('  Insufficient class balance for SVM training (need >= 2 samples per class).')

In [ ]:
# Cell 9: Summary
print('=' * 72)
print('  SUMMARY -- Week 4 Bayesian Optimisation (NN Surrogate)')
print('=' * 72)
print(f'  Function             : 2D Noisy ML Log-Likelihood')
print(f'  Objective            : Maximise')
print(f'  GP kernel            : RBF(length_scale=0.1, fixed)')
print(f'  NN architecture      : MLP [2 → 64 → 64 → 1], ReLU, Adam lr=1e-3, 1500 epochs')
print(f'  Acquisition          : NN gradient ascent (800 steps) vs GP UCB (beta=2.5, 30k grid)')
print(f'  Selected method      : {selected}')
print()
print(f'  Current best X*      : [{best_X[0]:.6f}, {best_X[1]:.6f}]')
print(f'  Current best Y*      : {best_Y:.6f}')
print()
print(f'  Next query point     : [{next_x[0]:.6f}, {next_x[1]:.6f}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 72)

## Week 4 -- Reflection

**Function**: F2 -- Noisy ML Log-Likelihood (2D)

### Query
- **Method**: NN gradient ascent from best known [1.0, 0.326531], with GP UCB (beta=2.5) as fallback.
- **Week 3 lesson**: [1,1] returned 0.036 — well below the best of 0.149 at [1.0, 0.326]. The peak is in the high-x1, mid-x2 region. Gradient ascent should refine the x2 coordinate.
- **Query type**: *(fill in after running -- Exploration or Exploitation?)*
- **Input submitted**: *(fill in portal submission string)*
- **Output received**: *(fill in after receiving result)*
- **Best so far**: *(update after result)*

### What this result taught us
*(fill in after receiving result)*

### Strategy for Week 5
*(fill in after reflecting on result)*